In [ ]:
%load_ext autoreload
%autoreload 2

### compute ellipse fit and ahull sizes

In [ ]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
import torch

from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")

In [ ]:
from utils import olc_client
c = olc_client.connect(verbose=True)

In [ ]:
import pandas as pd
import numpy as np
import scipy

import plotly.graph_objects as go
import matplotlib.pyplot as plt

from quan_propagation.func import count_col_ahull
from utils.ol_rf import compute_rf

# # Test if count_col is imported
# print(count_col)

In [ ]:
from utils.config import (
    CACHE_DIR, DATA_DIR, DATASET, FIG_DIR, HIT_THRE, N_FLOW, PARAMS_DIR, SIDE_CHAR,
)
DATA_DIR.mkdir(parents=True, exist_ok=True)

result_dir = FIG_DIR / 'quan_propagation'
result_dir.mkdir(parents=True, exist_ok=True)

cache_dir = CACHE_DIR / 'quan_propagation'
cache_dir.mkdir(parents=True, exist_ok=True)

# Some setups

## ol types

In [ ]:
# cell type table xlsx
oltypes = pd.read_excel(PARAMS_DIR / 'Nern-et-al_SuppTable01_Cell-types-and-counts.xlsx')
oltypes.rename(columns={'cell type':'cell_type', 'main groups':'main_groups'}, inplace=True)
print(oltypes.shape)

# only OL intrinsic types
oltypes_on = oltypes[oltypes.main_groups.isin(['ONIN','ONCN'])].copy()
print(oltypes_on.shape)


## rois

In [ ]:
# neuprint.fetch_primary_rois()
# rois_dict = neuprint.queries.fetch_roi_hierarchy(include_subprimary=False)
# rois_dict['CNS'].keys()

In [ ]:
from utils.query_roi import get_primary_rois
rois_cb = get_primary_rois('CentralBrain')
len(rois_cb)

## load prop matrix sum

In [ ]:
stepsn = scipy.sparse.load_npz(Path(DATA_DIR, f'{DATASET}_{SIDE_CHAR}_lat_flow_sum.npz'))
print(stepsn.shape)
print(Path(DATA_DIR, f'{DATASET}_{SIDE_CHAR}_lat_flow_sum.npz'))

## Get meta

In [ ]:
# LOAD Judith meta, this has AN's R7/8, 
meta = pd.read_csv(DATA_DIR / f'{DATASET}_meta.csv')
        
#  add superclass and class
meta.loc[meta['cell_type'].str.contains('^R7|^R8'), 'superclass'] = 'ol_sensory'
meta.loc[meta['cell_type'].str.contains('^R7|^R8'), 'class'] = 'visual'

print(meta.shape)  

In [ ]:
# make L1 excitatory so it's an ON cell 
meta.loc[meta.cell_type == 'L1', 'sign'] = 1

In [ ]:
# make dictionaries that map indices to meta info
idx_to_bodyId = dict(zip(meta.idx, meta.bodyId))
idx_to_coords = dict(zip(meta.idx, meta.coords))
bodyId_to_idx = dict(zip(meta.bodyId, meta.idx))

# Elliptical fit

In [ ]:
from utils.ol_rf import _compute_rf_params, _clean_rf_params

In [ ]:
# all_instances = pd.unique(
#     np.concatenate([
#         meta_ol['instance'].unique(),
#         meta_nonol['instance'].unique()
#     ])
# )

all_instances = pd.unique(
    np.concatenate([
        meta_ol['instance'].unique(),
        meta_nonol['instance'].unique()
    ])
)
len(all_instances)

### all cells with VIC > thr

In [ ]:
rf_fit = _compute_rf_params(
    meta, stepsn, 
    # in_instances=['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R7d_R', 'R8_R', 'R8d_R', 'HBeyelet_R'], 
    in_instances=['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R'], 
    out_instances = all_instances.tolist(),
    cumsum_thre = None,
)
rf_fit = _clean_rf_params(rf_fit)

# df_fit.rename(columns={'size': 'area'}, inplace=True)

print(rf_fit.shape)

# save
rf_fit.to_pickle(Path(cache_dir, 'rf_fit_thr10.pkl'))

# load
# rf_fit = pd.read_pickle(Path(cache_dir, 'rf_fit_thr10.pkl'))

In [ ]:
rf_fit = _compute_rf_params(
    meta, stepsn, 
    # in_instances=['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R7d_R', 'R8_R', 'R8d_R', 'HBeyelet_R'], 
    in_instances=['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R'], 
    out_instances = all_instances.tolist(),
    cumsum_thre = 0.7,
)
rf_fit = _clean_rf_params(rf_fit)

# df_fit.rename(columns={'size': 'area'}, inplace=True)

print(rf_fit.shape)

# save
rf_fit.to_pickle(Path(cache_dir, 'rf_fit_thr07.pkl'))

# load
# rf_fit = pd.read_pickle(Path(cache_dir, 'rf_fit_thr10.pkl'))

In [ ]:
rf_fit = pd.read_pickle(Path(cache_dir, 'rf_fit_thr10.pkl'))
rf_fit_2 = pd.read_pickle(Path(cache_dir, 'rf_fit_thr07.pkl'))

In [ ]:
# compare cumsum_thre
df = pd.merge(rf_fit[['instance','bodyId', 'size']], rf_fit_2[['bodyId','size']], on='bodyId', how='left', suffixes=('_thr10', '_thr07'))
# groupby instance and compute the median of ahull and size
df = df.groupby('instance').agg({'size_thr10':'median', 'size_thr07':'median'}).reset_index()
plt.figure(figsize=(6,6))
plt.scatter(df['size_thr10'], df['size_thr07'])
plt.plot([0, df['size_thr10'].max()], [0, df['size_thr10'].max()], 'r--')  # add y=x line for reference
plt.xlim(0, df['size_thr10'].max()*1.1)
plt.ylim(0, df['size_thr07'].max()*1.1)
plt.gca().set_aspect('equal')
plt.show()

# ahull area

volume under a 2D Gaussian within a radius of 1 std is about 39%

## vp + cb, right-side

In [ ]:
# right-size meta for vpn and vcbn with ht and vic
vp_cb_r = pd.read_pickle(Path(DATA_DIR, 'vp_cb_hit_vic_r.p'))

In [ ]:
# iterate, ver2, ~60 min
# meta_ahull = meta[meta.bodyId.isin(vp_cb_r.bodyId)].copy()

meta_ahull = vp_cb_r.copy()
print(meta_ahull.shape)

# filter by VIC, per instance 
thr_vic = 5e-4
inst = meta_ahull.groupby('instance').agg({'VIC':'median'}).reset_index()
inst = inst[inst.VIC > thr_vic]['instance']
meta_ahull = meta_ahull[meta_ahull.instance.isin(inst)]
print(meta_ahull.shape)

# # and per cell
# meta_ahull = meta_ahull[meta_ahull['VIC'] > thr_vic]
# print(meta_ahull.shape)

thr_mode = 'cumsum'
remove_frac = 0.6
meta_ahull['ahull_size'] = np.nan
meta_ahull['ahull_com_x'] = np.nan
meta_ahull['ahull_com_y'] = np.nan
meta_ahull['ahull_kept_points'] = np.nan
skipped_idx = []

inidx = meta.idx[meta.instance.isin(['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R'])]

for i, row in meta_ahull.iterrows():
    try:
        print(bodyId_to_idx[row['bodyId']])
        rf = compute_rf(meta, stepsn, inidx=inidx, outidx=bodyId_to_idx[row['bodyId']])
        df = rf.set_index('coords')[['effective weight']]
        edge_points, edges, area, com, kept_points = count_col_ahull(df, thr_mode=thr_mode, remove_frac = remove_frac)
        if edges is not None and len(edge_points) == 1:
            meta_ahull.at[i, 'ahull_size'] = area
            meta_ahull.at[i, 'ahull_com_x'] = com[0]
            meta_ahull.at[i, 'ahull_com_y'] = com[1]
            meta_ahull.at[i, 'ahull_kept_points'] = len(kept_points)
        else:
            skipped_idx.append(i)
    except ValueError as e:
        print(f"skipping bodyId {row['bodyId']}: {e}")
        skipped_idx.append(i)
        continue

meta_skipped = meta_ahull.loc[skipped_idx]

# save
# meta_ahull.to_pickle(Path(DATA_DIR, 'meta_ahull_cumsum60.pkl'))



## ol intrinsic, right-side

In [ ]:
meta_ahull = meta[meta['instance'].isin(oltypes_on['instance'])]

# # add ht and vic
# ht_r = pd.read_csv(DATA_DIR / f'{DATASET}_r_flow_{N_FLOW}step_{HIT_THRE}thre_hit.csv') # ht
# vic_r = pd.read_parquet(Path(DATA_DIR, f'{DATASET}_r_ol_cb_vic_raw.parquet')) # vic
# # combine
# meta_ahull = pd.merge(meta_ahull[['bodyId','cell_type','idx']], ht_r, on='idx', how='left')
# meta_ahull = pd.merge(meta_ahull, vic_r[['bodyId','VIC', 'main_groups', 'region']], on='bodyId', how='inner')
meta_ahull.shape

In [ ]:
# iterate, ver2, ~70 min

thr_mode = 'cumsum'
remove_frac = 0.6
meta_ahull['ahull_size'] = np.nan
meta_ahull['ahull_com_x'] = np.nan
meta_ahull['ahull_com_y'] = np.nan
meta_ahull['ahull_kept_points'] = np.nan
skipped_idx = []

inidx = meta.idx[meta.instance.isin(['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R'])]

for i, row in meta_ahull.iterrows():
    try:
        print(bodyId_to_idx[row['bodyId']])
        rf = compute_rf(meta, stepsn, inidx=inidx, outidx=bodyId_to_idx[row['bodyId']])
        df = rf.set_index('coords')[['effective weight']]
        edge_points, edges, area, com, kept_points = count_col_ahull(df, thr_mode=thr_mode, remove_frac = remove_frac)
        if edges is not None and len(edge_points) == 1:
            meta_ahull.at[i, 'ahull_size'] = area
            meta_ahull.at[i, 'ahull_com_x'] = com[0]
            meta_ahull.at[i, 'ahull_com_y'] = com[1]
            meta_ahull.at[i, 'ahull_kept_points'] = len(kept_points)
        else:
            skipped_idx.append(i)
    except ValueError as e:
        print(f"skipping bodyId {row['bodyId']}: {e}")
        skipped_idx.append(i)
        continue

meta_skipped = meta_ahull.loc[skipped_idx]

# save
# meta_ahull.to_pickle(Path(DATA_DIR, 'meta_ON_ahull_cumsum60.pkl'))

### DEBUG hex grid convension

In [ ]:
meta[meta.instance.eq('Mi1_R')].head()

In [ ]:
coords = meta.loc[
    meta.instance.eq('Mi1_R'),
    "coords"
].dropna()

xy = coords.str.split(",", expand=True).astype(int).values
# max and min of each col in xy
xy.max(axis=0), xy.min(axis=0)

In [ ]:
# plot coords
plt.figure(figsize=(6,6))
plt.scatter(xy[:,0], xy[:,1], s=10)
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In [ ]:
# meta_ahull[meta_ahull.instance.eq('T4c_R')].head()

In [ ]:
bodyId = 80483
rf = compute_rf(meta, stepsn, inidx=inidx, outidx=bodyId_to_idx[bodyId])
rf.head()

In [ ]:
fig = go.Figure()

rf_coords = rf.coords.str.split(",", expand=True).astype(int).to_numpy()
fig.add_trace(go.Scatter(
    x=rf_coords[:, 0],
    y=rf_coords[:, 1],
    mode='markers',
    name='meta coords',
    marker=dict(size=5, color='black'),
    customdata=rf_coords,
    hovertemplate='[%{customdata[0]}, %{customdata[1]}]<extra></extra>',
))

fig.add_trace(go.Scatter(
    x=rf.x,
    y=rf.y,
    mode='markers',
    name='rf',
    marker=dict(size=6, color='red', opacity=0.5),
    customdata=rf[['x', 'y']].values,
    hovertemplate='[%{customdata[0]}, %{customdata[1]}]<extra></extra>',
))

fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(width=600, height=600)
fig.show()

In [ ]:
edge_points, edges, area, com, kept_points = count_col_ahull(df, thr_mode='max', remove_frac = 0.5)

In [ ]:
area, com, kept_points

In [ ]:
import torch
import connectome_interpreter as ci

In [ ]:
df = rf.set_index('coords')[['effective weight']]

fig_rf0 = ci.hex_heatmap(
        df,
        custom_colorscale=[[0, "rgb(255, 255, 255)"], [1, "rgb(200, 20, 0)"]],
        global_min=0,
    )
if edges is not None and len(edge_points) == 1:
    fig_rf = go.Figure(fig_rf0)
    edge_pt = edge_points[0]
    for j in range(len(edge_pt)):
        p1, p2 = edge_pt[j], edge_pt[(j + 1) % len(edge_pt)]
        fig_rf.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], mode='lines',
            line=dict(color='blue', width=1), showlegend=False, hoverinfo='skip'))

fig_rf.show()

## Debug

In [ ]:
# load 
meta_ahull = pd.read_pickle(Path(DATA_DIR, 'meta_ahull_cumsum60.pkl'))
meta_ahull.shape

In [ ]:
# remove nan in ahull_size
meta_ahull = meta_ahull[meta_ahull['ahull_size'].notna()]
# add ratio of area to count
meta_ahull['area_to_count'] = meta_ahull['ahull_size'] / meta_ahull['ahull_kept_points']
# add count per instance group
meta_ahull['count'] = meta_ahull.groupby('instance')['bodyId'].transform('count')


In [ ]:
# scatter plot of area_to_count vs VIC with mouse hover showing instance
df_plt = meta_ahull.groupby('instance').agg(
    {'VIC':'median', 'area_to_count':'median', 'hitting_time':'median', 'count':'first', 'ahull_size':'median'}
    ).reset_index()
# df_plt = meta_ahull.copy()
# df_plt = df_plt[df_plt['hitting_time'] > 4]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_plt['VIC'],
    y=df_plt['area_to_count'],
    mode='markers',
    marker=dict(
        size=4,
        color=df_plt['ahull_size'],
        showscale=True,
        opacity=0.5,
    ),
    hovertemplate='instance=%{customdata[0]}<br>count=%{customdata[1]:.3g}<br>area=%{customdata[2]:.3g}<br>ht=%{customdata[3]:.3g}<extra></extra>',
    customdata=df_plt[['instance','count','ahull_size','hitting_time']].to_numpy(),
))
# log scale for x 
fig.update_layout(
    xaxis_type='log',
    xaxis_title='VIC',
    yaxis_title='Area to Count Ratio',
)

In [ ]:
# scatter plot of ahull_size vs hitting_time for main_groups == 'nonOL'
# Two VIC hues: blue for VIC < vic_cut, red for VIC >= vic_cut
vic_cut = 0.005
nonOL_data = meta_ahull[
    # (meta_ahull['main_groups'] == 'nonOL') &
    meta_ahull['ahull_size'].notna()
].copy()

low_vic = nonOL_data[nonOL_data['VIC'] < vic_cut]
high_vic = nonOL_data[nonOL_data['VIC'] >= vic_cut]

fig = go.Figure()

if len(low_vic) > 0:
    fig.add_trace(go.Scatter(
        x=low_vic['hitting_time'],
        y=low_vic['ahull_size'],
        mode='markers',
        name=f'VIC < {vic_cut}',
        marker=dict(
            size=4,
            color=low_vic['VIC'],
            colorscale='RdBu',
            showscale=True,
            colorbar=dict(title=f'VIC < {vic_cut}'),
            opacity=0.5,
        ),
        hovertemplate='instance=%{customdata[0]}<br>VIC=%{customdata[1]:.3g}<br>hitting_time=%{x}<br>ahull_size=%{y}<extra></extra>',
        customdata=low_vic[['instance', 'VIC']].to_numpy(),
    ))

# if len(high_vic) > 0:
#     fig.add_trace(go.Scatter(
#         x=high_vic['hitting_time'],
#         y=high_vic['ahull_size'],
#         mode='markers',
#         name=f'VIC >= {vic_cut}',
#         marker=dict(
#             size=4,
#             color=high_vic['VIC'],
#             colorscale='YlGn',
#             showscale=True,
#             colorbar=dict(title=f'VIC >= {vic_cut}'),
#             opacity=0.5,
#         ),
#         hovertemplate='instance=%{customdata[0]}<br>VIC=%{customdata[1]:.3g}<br>hitting_time=%{x}<br>ahull_size=%{y}<extra></extra>',
#         customdata=high_vic[['instance', 'VIC']].to_numpy(),
#     ))

fig.update_layout(
    title='',
    xaxis_title='hitting_time',
    yaxis_title='ahull_size',
)

fig.update_yaxes(type='log', range=[0, np.log10(1000)])
fig.show()

### low VIC + thr_vic -> disjoint point sets

In [ ]:
thr_mode, remove_frac = 'cumsum', 0.6
inidx = meta.idx[meta.instance.isin(['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R'])]    

In [ ]:
# meta[meta.instance == 'SMP399_c_R']
# meta_ahull[meta_ahull['instance'] == 'MeVP11_R']
# meta_ahull[meta_ahull['instance'] == 'LH002m_R']
meta_ahull[meta_ahull['instance'] == 'CB2948_R']

In [ ]:
bid = 47444
outidx=bodyId_to_idx[bid]

rf = compute_rf(meta, stepsn, inidx=inidx, outidx=outidx)
df = rf.set_index('coords')[['effective weight']]
edge_points, edges, area, com, kept_points = count_col_ahull(df, thr_mode=thr_mode, remove_frac = remove_frac)


In [ ]:
import torch
import connectome_interpreter as ci

In [ ]:
df = ci.result_summary(stepsn, inidx, outidx=outidx,
                    inidx_map = idx_to_coords, 
                    outidx_map = idx_to_bodyId,
                    display_threshold = 0,
                    display_output= False
                    )

In [ ]:

edge_points, edges, area, com, kept_points = count_col_ahull(
    df, thr_mode=thr_mode, remove_frac=remove_frac)

fig_rf0 = ci.hex_heatmap(
        df,
        custom_colorscale=[[0, "rgb(255, 255, 255)"], [1, "rgb(200, 20, 0)"]],
        global_min=0,
    )
if edges is not None and len(edge_points) == 1:
    fig_rf = go.Figure(fig_rf0)
    edge_pt = edge_points[0]
    for j in range(len(edge_pt)):
        p1, p2 = edge_pt[j], edge_pt[(j + 1) % len(edge_pt)]
        fig_rf.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], mode='lines',
            line=dict(color='blue', width=1), showlegend=False, hoverinfo='skip'))

fig_rf.show()

In [ ]:
fig = ci.hex_heatmap(df)
fig

# End